# Пример разбиения текста и извлечения чанков
---



In [1]:
!pip install httpcore[socks]
!pip install httpx[socks]

## Описание задачи
---
> На вход поступает маркированный текст.

> На выходе получаем описание чанков, на которые разбит текст.

Используемые входные ресурсы:

- **marked_text**: MarkedTextClass, маркированный текст; Название разделов разного уровня определяется наличием в начале строки решеток. Решетка, после которой следует пробел и потом заголовок - это заголовок 1-го уровня. Если 2 решетки - то заголовок 2-го уровня и т.д. Название документа - это текст после 1-й решетки. Нас будут интересовать Разделы, заголовки которых идут после 2-х решеток.

Выходные ресурсы (артефакты):
- **chunks** = List[**ChunkClass**], список выделенных в тексте чанков; каждый чанк содержит информацию:
- - chunk_id, идентификатор чанка;
- - serial_num, сквозной номер в тексте;
- - abstract, краткое содержание;
- - content, текст чанка;
- - extent, размер чанка в символах;
- - source, идентификатор текста, из которого взят чанк.



In [2]:
from typing import List, Union, Literal, Annotated, Dict, Type, Any
from annotated_types import MaxLen, Le, MinLen
from pydantic import BaseModel, Field
from datetime import date


class MarkedTextClass(BaseModel):
    """ размеченный Markdown текст (документ или фрагмент) для анализа """
    text: str = Field(..., description="размеченный текст")


class ChunkClass(BaseModel):
    """ извлеченный запрос """
    title: str = Field(..., description="полное название раздела текста, из которого взят чанк")
    serial_num: int = Field(..., description="сквозной номер чанка в документе")
    abstract: str = Field(..., description="краткое содержание чанка")
    content: str = Field(..., description="содержимое чанка")
    extent: int = Field(..., description="размер чанка в символах")
    source: Union[str, None] = Field(..., description="идентификатор текста, из которого взят чанк")


class ChunkAnalisysResultClass(BaseModel):
    chunk_list: List[ChunkClass] = Field(..., description="список чанков текста")
    resume: str = Field(..., description="резюме по решению задачи")

## Готовим исходные данные
---
Загрузим текст из файла *.md и создадим соответствующий артефакт

In [25]:
# подключаемся к приводу
from pprint import pprint
import os
from google.colab import drive
drive.mount('/content/drive')

# переходим в доменную корневую папку
# Скорректированный путь: обычно MyDrive (с большой М и без пробела) в Colab
path = "/content/drive/MyDrive/Colab Notebooks/Moneta/KBase/domains/abitura/"

if not os.path.exists(path):
    print(f"Ошибка: Путь '{path}' не найден. Проверьте структуру папок на Google Диске.")
    parent_dir = os.path.dirname(path.rstrip('/')) # Get the parent directory for listing
    print(f"Содержимое каталога '{parent_dir}':")
    if os.path.exists(parent_dir):
        pprint(os.listdir(parent_dir))
    else:
        print(f"Каталог '{parent_dir}' также не существует.")
    raise FileNotFoundError(f"Каталог '{path}' не существует.")

os.chdir(path)
docfiles = os.listdir("./docs")
pprint(docfiles)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['documents_required_for_admission.md']


In [28]:
# загрузим любой маркированный текст документа
filename = docfiles[0]
filepath = os.path.join("./docs", filename)
with open(filepath, "r", encoding="utf-8") as f:
  doc_text = f.read()
  print(f"Loaded extracted text from {filepath}; {len(doc_text)} chars")

# создадим объект класса MarkedText
marked_text = MarkedTextClass(text=doc_text)

Loaded extracted text from ./docs/documents_required_for_admission.md; 5781 chars


In [29]:
print(marked_text)

text='# Документы, требуемые при поступлении на разные направления подготовки и специальности высшего и среднего образования\n\nЗдесь описаны документы, которые требуются при подаче заявлений на поступление на разные уровни подготовки.\n\n## Бакалавриат\n\n1. Предъявить паспорт при подаче заявления + ксерокопия разворота с фото, с пропиской (не заверенные).\n\n2. Предъявить оригинал или копия документов об образовании:\n   - На базе 11 классов — аттестат о среднем общем образовании с вкладышем.\n   - На базе СПО:\n     - при обучении в учреждениях СПО на базе 9 классов — диплом с вкладышем;\n     - при обучении в учреждениях СПО на базе 11 классов — диплом с вкладышем + аттестат о среднем общем образовании с вкладышем.\n   - На базе НПО — диплом с записью о получении среднего (полного) общего образования.\n   - При поступлении на второе высшее образование — диплом с вкладышем о первом высшем образовании.\n\n3. Фотографии 6 штук 3×4.\n\n4. Предъявить СНИЛС.\n\n5. Результаты ЕГЭ (если ес

In [30]:
docfiles = os.listdir()
pprint(docfiles)

['docs', 'chunking.ipynb', 'src', 'chunks']


## Config агента
---

In [18]:
# собираем конфигурацию агента
import openai
from openai import OpenAI
import src.envs

openai.api_key = os.getenv("OPENAI_API_KEY")

""" конфигурационные параметры агента для GPT-модели """
chunk_gpt_agent_config = {
    "config_name": "chunk_analysis_GPT_Agent", # название конфигурации
    "api_key": openai.api_key,  # API ключ для доступа к модели GPT
    "http_proxy": os.getenv("HTTP_PROXY"),  # если используется прокси сервер
    "https_proxy": os.getenv("HTTPS_PROXY"), # если используется прокси сервер
    "model": "gpt-5-mini-2025-08-07", # Название используемой модели GPT
    "instruction": """ Ты являешься экспертом по разбиению текстов на чанки.
    Твоя задача — разбить исходный текст на идущие последовательно разделы (chapters)
    и чанки (chunks) - смысловые фрагменты, состоящие из одного или несколько абзацев.

    Содержание текста разбивается на идущие последовательно неперекрывающиеся чанки.
    Старайся не разделять на разные чанки единый смысловой фрагмент (нумерованный список, перечисление, таблица, ...).

    Каждый объект Chunk должен содержать: 'title', 'serial_num', 'abstract', 'content', 'extent', 'source'.
    Атрибут 'source' содержит начальный заголовок текста.

    В завершении работы напиши свое резюме по решению задачи.
    Если не удалось выполнить задание, объясни в резюме причины невыполнения.

    Формат выходных данных должен быть JSON и содержать описание результатов как объект класса ChunkAnalisysResultClass
    """
}

#  ВАЖНО! Заголовки, которые находятся внутри текста, являются естественными границами чанков и должны быть началом чанка.


print(chunk_gpt_agent_config['instruction'])

 Ты являешься экспертом по разбиению текстов на чанки.
    Твоя задача — разбить исходный текст на идущие последовательно разделы (chapters)
    и чанки (chunks) - смысловые фрагменты, состоящие из одного или несколько абзацев.

    Содержание текста разбивается на идущие последовательно неперекрывающиеся чанки.
    Старайся не разделять на разные чанки единый смысловой фрагмент (нумерованный список, перечисление, таблица, ...).

    Каждый объект Chunk должен содержать: 'title', 'serial_num', 'abstract', 'content', 'extent', 'source'.
    Атрибут 'source' содержит начальный заголовок текста.

    В завершении работы напиши свое резюме по решению задачи.
    Если не удалось выполнить задание, объясни в резюме причины невыполнения.

    Формат выходных данных должен быть JSON и содержать описание результатов как объект класса ChunkAnalisysResultClass
    


## ChunkAnalysisAgent
---
Реализовать агента для разбиения текста на разделы и чанки.

Описание

> Необходимо спроектировать и реализовать агента, который принимает на вход размеченнный текст MarkedText, разбивает его на разделы и чанки и формирует описание получившихся чанков (ChunkClass).


In [19]:
from openai import OpenAI
import json

class ChunkAnalysisAgent:

    def __init__(self, config: dict):
        self.config = config
        self.client = OpenAI(api_key=self.config['api_key'])
        self.marked_text = None
        self.chunk_analysis_result = None

    def set_text(self, marked_text):
        self.marked_text = marked_text

    def solve_task(self):
        if self.marked_text is None:
            raise ValueError("Текст не задан")

        # 1. Get input data
        marked_text_obj = self.marked_text

        # 2. Construct the prompt
        system_instruction = self.config['instruction']
        prompt_message = f"""
{system_instruction}

Исходный размеченный текст для анализа:
---
{marked_text_obj.text}
---

Ожидаемый формат выходных данных (JSON Schema) должен соответствовать классу ChunkAnalisysResultClass:
---
{ChunkAnalisysResultClass.model_json_schema()}
---

Поле `serial_num` для `ChunkClass` должно быть строкой и числом соответственно.
Поле `title` для `ChunkClass` должно быть заголовком раздела, из которого взят чанк.
"""
        # 3. Call the LLM
        try:
            chat_completion = self.client.chat.completions.create(
                model=self.config['model'],
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": prompt_message}
                ],
                response_format={"type": "json_object"}
            )
            llm_output_str = chat_completion.choices[0].message.content
            print(f"LLM Raw Output: {llm_output_str}") # For debugging

            # 4. Parse and validate the LLM's response
            llm_response_dict = json.loads(llm_output_str)

            # Validate the entire result against ChunkAnalisysResultClass
            self.chunk_analysis_result = ChunkAnalisysResultClass(**llm_response_dict)

            # print resume
            print(f"Выполнение задачи завершено:\n {self.chunk_analysis_result.resume}")

        except Exception as e:
            print(f"Ошибка при анализе чанков: {e}")


    def __call__(self):
        self.solve_task()

In [31]:
# Создаем экземпляр ChunkAnalysisAgent
chunk_analysis_agent = ChunkAnalysisAgent(chunk_gpt_agent_config)
print("Агент ChunkAnalysisAgent создан.")

# Назначаем задачу агенту
chunk_analysis_agent.set_text(marked_text)

print("Агент ChunkAnalysisAgent создан и задача назначена.")

Агент ChunkAnalysisAgent создан.
Агент ChunkAnalysisAgent создан и задача назначена.


## Решаем ЗАДАЧУ

In [32]:
# Запускаем решение задачи агентом
chunk_analysis_agent.solve_task()

LLM Raw Output: {
  "chunk_list": [
    {
      "title": "Документы, требуемые при поступлении на разные направления подготовки и специальности высшего и среднего образования",
      "serial_num": 1,
      "abstract": "Краткое вступление: указано, что далее описаны требуемые документы при подаче заявлений на разные уровни подготовки.",
      "content": "Здесь описаны документы, которые требуются при подаче заявлений на поступление на разные уровни подготовки.",
      "extent": 107,
      "source": "Документы, требуемые при поступлении на разные направления подготовки и специальности высшего и среднего образования"
    },
    {
      "title": "Бакалавриат",
      "serial_num": 2,
      "abstract": "Список документов, необходимых для поступления на бакалавриат: паспорт, документы об образовании (с вариантикой по базовой подготовке), фото, СНИЛС, результаты ЕГЭ и документы, подтверждающие льготы, целевое обучение, медицинская справка для отдельных специальностей и прочие подтверждающие до

## Анализ РЕШЕНИЯ
---

In [33]:
chunk_analysis_agent.chunk_analysis_result.resume

'Резюме: Текст разбит на последовательные, неперекрывающиеся смысловые разделы (чанки). В качестве единиц взяты: вводная (вступление), целиком каждый раздел/уровень подготовки (Бакалавриат, Магистратура, Среднее профессиональное образование, Комплект документов для поступления в аспирантуру) и отдельно выделено примечание. При этом внутри разделов сохранены целые нумерованные списки и вложенные перечисления — я не разбивал отдельные пункты списка на отдельные чанки, чтобы не дробить единый смысловой фрагмент (как было указано в инструкции). Поле "title" у каждого чанка указано как заголовок раздела, из которого взят чанк; поле "source" — как начальный заголовок всего текста. Значения поля "extent" (размер чанка в символах) приведены как оценочные (приближённые) величины, поскольку точный подсчёт символов каждой строки вручную был бы очень трудоёмким и мог бы ввести в документ неожиданные ошибки форматирования; при необходимости могу пересчитать точные значения программно и предоставить

In [34]:
import os
import json

chunks_path = "./chunks/"

# Ensure the CQs directory exists
os.makedirs(chunks_path, exist_ok=True)

# Get the base name of the original document without extension
base_filename = "chunks_" + os.path.splitext(os.path.basename(filename))[0]

# Construct the output file path
output_filepath = os.path.join(chunks_path, f"{base_filename}.json")

# Convert the CQListCLass object to a JSON string
chunk_analisys_result = chunk_analysis_agent.chunk_analysis_result
if chunk_analisys_result is not None:
    json_data = chunk_analisys_result.model_dump_json(indent=2)

    # Save the JSON data to the file
    with open(output_filepath, "w", encoding="utf-8") as f:
        f.write(json_data)

    print(f"Содержимое chunk_analisys_result успешно сохранено в файл: {output_filepath}")
else:
    print("Ошибка: Нет данных для сохранения. 'chunk_analisys_result' не найден или пуст.")

Содержимое chunk_analisys_result успешно сохранено в файл: ./chunks/chunks_documents_required_for_admission.json
